In [5]:
import pandas as pd
import numpy as np
import pickle

In [17]:
! pip install wandb

  Obtaining dependency information for wandb from https://files.pythonhosted.org/packages/14/1c/faf90318ff4bcb23d533b32ca3a9db6616532f312563f46d02a0c121ecff/wandb-0.17.3-py3-none-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
  Obtaining dependency information for docker-pycreds>=0.4.0 from https://files.pythonhosted.org/packages/f5/e8/f6bd1eee09314e7e6dee49cbe2c5e22314ccdb38db16c9fc72d2fa80d054/docker_pycreds-0.4.0-py2.py3-none-any.whl.metadata
  Obtaining dependency information for gitpython!=3.1.29,>=1.0.0 from https://files.pythonhosted.org/packages/e9/bd/cc3a402a6439c15c3d4294333e13042b915bbeab54edc457c723931fed3f/GitPython-3.1.43-py3-none-any.whl.metadata
  Obtaining dependency information for protobuf!=4.21.0,<6,>=3.19.0 from https://files.pythonhosted.org/packages/27/e4/8dc4546be46873f8950cb44cdfe19b79d66d26e53c4ee5e3440406257fcd/protobuf-5.27.2-cp38-abi3-manylinux2014_x86_64.whl.metadata
  Obtaining dependency information for sen

In [18]:
import wandb
wandb.login(key='755eda0bfebf5cc95b8de1b0c1b74e6210554916')

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: alexander_schubert. Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /wynton/protected/home/ibrahim/alex_schubert/.netrc


True

### Load train data

In [6]:
# Load the DataFrame from the pickle file
human_label_df = pd.read_pickle('../data/human_label_data.pkl')

# Display the first few rows of the DataFrame to verify it loaded correctly
display(human_label_df.head())

# You can also check the shape of the DataFrame
print(human_label_df.shape)

,curve_idx,target,cycle_no,rn,Fn,igi_call,split
0,24768,S gene,1,147103.828125,147103.83,Negative,val
1,24768,S gene,2,146864.656250,146864.66,Negative,val
2,24768,S gene,3,146410.109375,146410.11,Negative,val
3,24768,S gene,4,146188.328125,146188.33,Negative,val
4,24768,S gene,5,146078.421875,146078.42,Negative,val


(1910490, 7)


### Label data

In [ ]:
print(f"label data shape {human_label_df.shape}")

# curve_data = pd.read_hdf('data/new_data_1.h5',key = 'curve_data')
# print(f"curve data shape {curve_data.shape}")

# print(len(curve_data['curve_idx'].unique()))
print(len(human_label_df['curve_idx'].unique()))

# is_subset = set(groundtruth_df['curve_idx'].unique()).issubset(set(curve_data['curve_idx'].unique()))
# print(is_subset)

# print(curve_data['curve_idx'].max())
# print(groundtruth_df['curve_idx'].max())


#print(curve_data.head(50))
human_label_df = human_label_df.sort_values(by=['curve_idx', 'cycle_no'])
#print(human_label_df.head(50))

###########################################
## Create dataframe with PCR curve labels
###########################################

### We should use the groundtruth column as a training label

#Get target dataframe
target_data = human_label_df.groupby('curve_idx').tail(1)
print(target_data.columns)
target_data['Igi_call_quant'] = (target_data['igi_call']=="Positive").astype(int)
# target_data['groundtruth_target'] = (target_data['groundtruth']==1).astype(int)
print(target_data.sort_values(by="curve_idx").head())
print(target_data['split'].unique())
print(target_data.shape)

target_data.to_csv('../data/human_label_df_target_data_split_v1.csv')
# target_data.to_csv('data/new_groundtruth_df_target_data_no_invalid.csv')
display(target_data.head())

### Create Curve Data

In [19]:
print(human_label_df.shape)
print(human_label_df.head(5))

(1910490, 7)
         curve_idx  target  cycle_no            rn         Fn  igi_call split
1513120          0  E gene         1  262576.65625  262576.66  Negative  test
1513121          0  E gene         2  262600.18750  262600.20  Negative  test
1513122          0  E gene         3  262856.37500  262856.38  Negative  test
1513123          0  E gene         4  263037.81250  263037.80  Negative  test
1513124          0  E gene         5  263639.18750  263639.20  Negative  test


In [21]:
from tqdm import tqdm

####################################
## Create dictionary of PCR curves
####################################

print(human_label_df.shape)
curve_ids = human_label_df['curve_idx'].unique()

# Initialize an empty dictionary
curve_dict_fn = {}
curve_dict_drn = {}

# Initialize variables to track the current cycle and array
current_cycle = None
current_array = None

# Iterate over each row in the DataFrame
for index, row in tqdm(human_label_df.iterrows()):
    curve_idx = row['curve_idx']
    cycle_no = row['cycle_no']
    Fn = row['Fn']
    drn = row['rn']

    # Create a new array if cycle_no resets to 1
    if cycle_no == 1:
        current_array_fn = []
        current_array_drn = []
        
    # Append Fn to the current array
    current_array_fn.append(Fn)
    current_array_drn.append(drn)

    # Save the array in the dictionary with key as curve_idx
    curve_dict_fn[curve_idx] = current_array_fn.copy()
    curve_dict_drn[curve_idx] = current_array_drn.copy()

# # Open the file in read-binary mode and load the dictionary
# with open('data/groundtruth_df_curve_dict_split_v3.pkl', 'wb') as file:
#     pickle.dump(curve_dict, file)
# groundtruth_curve_dict_fn = {key: curve_dict_fn[key] for key in groundtruth_ids}
# groundtruth_curve_dict_drn = {key: curve_dict_drn[key] for key in groundtruth_ids}

# # # Open the file in read-binary mode and load the dictionary
# with open('data/new_groundtruth_df_curve_dict_fn_no_invalid.pkl', 'wb') as file:
#     pickle.dump(groundtruth_curve_dict_fn, file)

# with open('data/new_groundtruth_df_curve_dict_drn_no_invalid.pkl', 'wb') as file:
#     pickle.dump(groundtruth_curve_dict_drn, file)

with open('../data/human_label_curve_dict_fn.pkl', 'wb') as file:
    pickle.dump(curve_dict_fn, file)

with open('../data/human_label_curve_dict_drn.pkl', 'wb') as file:
    pickle.dump(curve_dict_drn, file)

(1910490, 7)


0it [00:00, ?it/s]

1910490it [02:03, 15409.32it/s]


In [23]:
# Define a function for generating and saving a single image
import pandas as pd
import pickle as pkl
import os

import matplotlib.pyplot as plt

from concurrent.futures import ProcessPoolExecutor
from tqdm.contrib.concurrent import process_map  # For progress bar with multiprocessing

def save_curve_as_image(curve_idx):
    sequence = curve_dict[curve_idx][:40]
    imgs_folder = '../data/curve_img_human_rn'
    if not os.path.exists(imgs_folder):
        os.makedirs(imgs_folder, exist_ok=True)  # Ensure thread-safe directory creation
    plt.plot(sequence, linewidth=6)
    plt.axis('off')  # This will turn off the axis labels and ticks
    plt.savefig(f'{imgs_folder}/curve_{curve_idx}.png')
    plt.close()

with open('../data/human_label_curve_dict_drn.pkl', 'rb') as file: # 'data/karlen_curve_dict.pkl' new_full_curve_dict_fn_v1.pkl
    curve_dict = pkl.load(file)

#target_df = pd.read_csv('data/groundtruth_df_target_data_split_v2.csv')
target_df = pd.read_csv('../data/human_label_df_target_data_split_v1.csv') # 'data/karlen_target_data.csv' new_groundtruth_df_target_data_v1.csv

###########################################
## Get the right normalization values
###########################################

target_df_filtered = target_df[target_df['split']=='train']
#target_df_filtered = target_df[target_df['split']=='test']
curve_dict_filtered = {k: curve_dict[k] for k in curve_dict.keys() if k in target_df_filtered['curve_idx'].values}

mean_list = []
std_list = []

for key, curve in tqdm(curve_dict_filtered.items()):
    mean_curve = np.array(curve).mean().item()
    std_curve = np.array(curve).std().item()

    mean_list.append(mean_curve)
    std_list.append(std_curve)

norm_mean = np.array(mean_list).mean().item()
norm_std = np.array(std_list).mean().item()

###########################################
## Save curves as images
###########################################

#imgs_folder = 'data/curve_imgs_new_cleaner'
curve_indices = list(curve_dict.keys())

# Using ProcessPoolExecutor to parallelize the loop
# Adjust `max_workers` as per your system's CPU resources if necessary
with ProcessPoolExecutor(max_workers=os.cpu_count()//2) as executor:
    # Wrap with tqdm for a progress bar
    list(process_map(save_curve_as_image, curve_indices, chunksize=10, max_workers=os.cpu_count()//2))

100%|██████████| 30835/30835 [00:01<00:00, 22994.47it/s]


  0%|          | 0/44027 [00:00<?, ?it/s]

### Load ViT weights  

In [16]:
import torch
import torchvision.models as models

# Specify the model
model_name = 'vit_b_32'
weights = 'IMAGENET1K_V1'

# Create a temporary model to trigger the download
temp_model = getattr(models, model_name)(weights=weights)

# Get the state dict
state_dict = temp_model.state_dict()

# Save the state dict
torch.save(state_dict, f'../vit_weights/{model_name}_{weights}.pth')

print(f"Weights saved to {model_name}_{weights}.pth")

Weights saved to vit_b_32_IMAGENET1K_V1.pth
